In [ ]:
# Split raw train data into stratified train/val (mixed amateur / expert)

from pathlib import Path
import numpy as np
import gzip
import pickle
import os
from sklearn.model_selection import train_test_split

# Adjust BASE_PATH if needed depending on where this notebook is located
# Here we assume the notebook is in `notebooks-lm/` and project root is the parent directory.
BASE_PATH = Path.cwd().parent
RAW_PATH = BASE_PATH / "data" / "raw" / "train.pkl"
SPLIT_DIR = BASE_PATH / "data" / "splitted"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Base path:", BASE_PATH)
print("Raw train path:", RAW_PATH)
print("Output dir:", SPLIT_DIR)


In [ ]:
def load_zipped_pickle(filename: str):
    with gzip.open(filename, "rb") as f:
        return pickle.load(f)


def save_zipped_pickle(obj, filename: str):
    with gzip.open(filename, "wb") as f:
        pickle.dump(obj, f, protocol=2)
    print(f"Saved {len(obj)} samples to {filename}")


In [ ]:
# Load full raw train set (videos)
data = load_zipped_pickle(str(RAW_PATH))
print(f"Loaded {len(data)} samples from {RAW_PATH}")

# Build label list for stratification (e.g. "amateur" / "expert")
labels = [item.get("dataset") for item in data]
unique, counts = np.unique(labels, return_counts=True)
print("Overall dataset distribution:")
for u, c in zip(unique, counts):
    print(f"  {u}: {c}")


In [ ]:
VAL_RATIO = 0.2
SEED = 42

# Stratified train/val split (mixed amateur / expert), preserving dataset proportions
train_data, val_data = train_test_split(
    data,
    test_size=VAL_RATIO,
    random_state=SEED,
    shuffle=True,
    stratify=labels,
)

print("Total samples:", len(data))
print("Train samples:", len(train_data))
print("Val samples  :", len(val_data))

# Optional: check distributions
from collections import Counter

def count_by_dataset(items):
    return Counter([it.get("dataset") for it in items])

print("Train distribution:", count_by_dataset(train_data))
print("Val distribution  :", count_by_dataset(val_data))


In [ ]:
# Save splits to data/splitted (mixed amateur / expert)

save_zipped_pickle(train_data, str(SPLIT_DIR / "train_split.pkl"))
save_zipped_pickle(val_data,   str(SPLIT_DIR / "val_split.pkl"))
